In [ ]:
# !pip install --upgrade transformers datasets evaluate

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score,
)
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from datasets import load_dataset
import evaluate

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

TEST_SIZE = 0.2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 2
BATCH_SIZE = 16

DATA_PATH = "data/sentence_sets_trimmed.csv"
LABELS = {"female": 0, "male": 1}

LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

STRATIFY_ENABLED = True
DEGENDER_ENABLED = False
BALANCED_WEIGHTS_ENABLED = False

MAX_LENGTH = 512

OUTPUT_NAME = f"{MODEL_NAME}-finetuned-nlp-letters-{TEXT_COLUMN}"
OUTPUT_NAME += "-degendered" if DEGENDER_ENABLED else ""
OUTPUT_NAME += "-stratified-" if STRATIFY_ENABLED else ""
OUTPUT_NAME += "-balanced" if BALANCED_WEIGHTS_ENABLED else ""

In [ ]:
class LettersBERTModule(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=len(LABELS), class_weights=None, cls_or_mean="cls"):
        super().__init__()

        self.config = AutoConfig.from_pretrained(model_name)
        self.config.num_labels = num_labels
        self.config.class_weights = class_weights
        self.config.cls_or_mean = cls_or_mean

        # Layers
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.classifier = nn.Linear(self.config.hidden_size, self.config.num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Either use [CLS] token or mean pooling
        if self.config.cls_or_mean == "mean":
            output = torch.mean(outputs.last_hidden_state, dim=1)
        else:
            output = outputs.last_hidden_state[:, 0, :]

        # Logits and loss
        logits = self.classifier(output)
        loss = None

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.config.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def degender(text: str) -> str:
    """
    Replace gender pronouns and nouns with neutral gender.
    """
    # For clarity, we embed the dictionaries right here
    degender_pronouns = {
        " mr ": " mx ",
        " mrs ": " mx ",
        " ms ": " mx ",
        " miss ": " mx ",
        " mister ": " mx ",
    }
    degender_nouns = {
        " man ": " person ",
        " men ": " persons ",
        " woman ": " person ",
        " women ": " persons ",
        " man's ": " person's ",
        " men's ": " person's ",
        " woman's ": " person's ",
        " women's ": " person's ",
        " gentleman ": " person ",
        " lady ": " person ",
        " gentleman's ": " person's ",
        " lady's ": " person's ",
    }

    text = text.lower()

    for old, new in degender_pronouns.items():
        text = text.replace(old, new)

    for old, new in degender_nouns.items():
        text = text.replace(old, new)

    return text

In [ ]:
def preprocess(data):
    texts = data[TEXT_COLUMN]

    if DEGENDER_ENABLED:
        texts = [degender(t) for t in texts]

    tokenized = tokenizer(
        texts, 
        truncation=True, 
        padding=True
    )

    labels = [LABELS[label_str] for label_str in data[LABEL_COLUMN]]
    tokenized["labels"] = labels

    return tokenized

In [ ]:
dataset = load_dataset("csv", data_files=DATA_PATH)

dataset = dataset["train"].train_test_split(
    test_size=TEST_SIZE,
    stratify_by_column=LABEL_COLUMN if STRATIFY_ENABLED else None,
    seed=100,
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
confusion_metric = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    precision = precision_metric.compute(predictions=preds, references=labels, average="macro")
    recall = recall_metric.compute(predictions=preds, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")

    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)

    cr = classification_report(labels, preds, target_names=LABELS.keys())
    cm = confusion_metric.compute(predictions=preds, references=labels)

    print("Confusion Matrix:\n", cm["matrix"])
    print("Classification Report:\n", cr)
    print("MCC:", mcc)
    print("Balanced Accuracy:", bal_acc)

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"],
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
    }

In [ ]:
# Class weight balancing
class_weights = torch.tensor(
    compute_class_weight(
        "balanced", 
        classes=np.unique(train_dataset[LABEL_COLUMN]), 
        y=train_dataset[LABEL_COLUMN]
    ), dtype=torch.float
)

In [ ]:
# model = LettersBERTModule(model_name=MODEL_NAME, num_labels=len(LABELS), class_weights=class_weights)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, class_weights=class_weights)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./{OUTPUT_NAME}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()